# 04 · Missingness features: does knowing WHAT was measured help?

After forward fill, the model cannot tell a fresh measurement from a value copied forward from hours ago. The data analysis (notebook 01) also showed that septic patients get measured more often. Two kinds of features give that information back, for every vital sign and lab value:

- **flag** `X_measured`: 1 if X was really measured in this hour, else 0
- **hours since** `X_hours_since`: how many hours ago X was last measured (capped at 48; "never measured yet" also counts as 48)

Three feature sets, same 5-fold patient-level CV on 80% of Hospital A, same weighted logistic regression (notebook 03). Only the inputs change, so any difference comes from them:

| feature set | inputs |
|---|---|
| baseline | the filled values only |
| flags | baseline + flags |
| flags + hours since | baseline + flags + hours since |

In [1]:
import sepsis as s

data_A = s.load_hospital("A")
dev_ids, test_ids = s.split_patients(data_A, test_size=0.20)      # same 80/20 split as notebook 03
fold_of = s.make_folds(data_A, dev_ids)

# missingness=True (the default) also builds the flag and hours-since columns, once for everybody.
# They only look back in time within one patient, so building them before the folds leaks nothing.
dev = s.prepare_dev(data_A, dev_ids, fold_of)
print(f"{dev['flags'].shape[1]} flag columns, {dev['since'].shape[1]} hours-since columns")

34 flag columns, 34 hours-since columns


In [2]:
oofs = {
    "baseline": s.run_cv(dev, s.ALL_FEATURES),
    "flags": s.run_cv(dev, s.ALL_FEATURES, flags=True),
    "flags + hours since": s.run_cv(dev, s.ALL_FEATURES, flags=True, since=True),
}

trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 66 input features
trained on 66 input features
trained on 66 input features
trained on 66 input features
trained on 66 input features
trained on 96 input features
trained on 96 input features
trained on 96 input features
trained on 96 input features
trained on 96 input features


## 1. AUROC and AUPRC

Read the fold-by-fold differences to the baseline first: a gain that shows up in every fold is real, while a difference smaller than its standard deviation across folds is probably noise.

In [3]:
summary, diffs = s.compare_auc(oofs)
display(summary.round(4))
display(diffs.round(4))

AUROC           AUPRC        
                       mean     std    mean     std
Features                                           
baseline             0.7462  0.0230  0.0827  0.0142
flags                0.7647  0.0207  0.0873  0.0143
flags + hours since  0.7847  0.0232  0.0930  0.0159

AUROC                       AUPRC                    
Features   flags flags + hours since   flags flags + hours since
1         0.0167              0.0403  0.0047              0.0121
2         0.0219              0.0369  0.0072              0.0105
3         0.0189              0.0419  0.0020              0.0071
4         0.0138              0.0355  0.0039              0.0122
5         0.0209              0.0380  0.0051              0.0095
mean      0.0185              0.0385  0.0046              0.0103
std       0.0033              0.0026  0.0019              0.0021

## 2. Each feature set at its own best cutoff

Each set gets the cutoff (0.10 to 0.95) with the highest clinical utility on the pooled out-of-fold predictions. Adding features can shift the probabilities, so one fixed cutoff for all three sets would be unfair to some of them.

In [4]:
s.best_cutoff_table(oofs).round(4)

,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
baseline,0.55,0.1094,0.0608,0.5454,0.1871,0.3099,0.3098,0.0379
flags,0.55,0.1139,0.0631,0.5826,0.1920,0.3424,0.3424,0.0409
flags + hours since,0.55,0.1179,0.0652,0.6171,0.1966,0.3674,0.3673,0.0428


## Conclusion

Both missingness features help, and the gain is consistent: across the 5 folds, AUROC rises from about 0.746 (baseline) to 0.765 with the flags and 0.785 with flags + hours since, and AUPRC from about 0.083 to 0.087 and 0.093. Every fold improves, and the fold-to-fold spread of the gain is far smaller than the gain itself, so this is not noise. Hours since adds about as much again as the flags alone: *how long ago* something was measured matters, not only *whether* it was measured this hour.

A caution from notebook 01: how often tests are taken partly reflects hospital routine. These features may carry over to Hospital B less well than the physiological values do, which the final evaluation will show. The 20% test set and Hospital B are kept untouched.